# 🐍 Week 1: Async Python, Pydantic & REST APIs for AI Agents

**Goal:** Master the Python foundations you need BEFORE building any AI agent.

**Why this matters:** Every AI agent framework (LangChain, AutoGen, CrewAI) is built on these patterns. If you understand them deeply, you can:
- Debug any agent issue
- Build custom agents without frameworks
- Extend any framework confidently

---

| Section | Topic | Agent Relevance |
|---------|-------|-----------------|
| 1 | Async Python (`asyncio`) | Concurrent tool calls, streaming |
| 2 | Pydantic & Type Hints | Structured LLM outputs, tool schemas |
| 3 | Decorators for Tools | How frameworks register agent tools |
| 4 | Error Handling & Retry | Robust agents that recover from failures |
| 5 | REST APIs & HTTP | Every tool call = API call |
| 6 | JSON Schemas | LLM function calling, tool definitions |
| 7 | Capstone: AsyncAPIClient | Build an agent's tool execution layer |

In [ ]:
# Install dependencies
!pip install pydantic aiohttp httpx tenacity jsonschema

---
# Section 1: Async Python (`asyncio`) for AI Agents

## Why Async Matters for Agents

When an AI agent runs, it often needs to:
- Call an LLM API (takes 1-5 seconds)
- Execute multiple tools in parallel (web search + database query + file read)
- Stream responses token-by-token to the user
- Handle timeouts and cancellation

**Without async:** Each operation blocks — 5 API calls × 2 seconds = 10 seconds  
**With async:** All 5 run concurrently — total ≈ 2 seconds

```
Sequential (Synchronous):
  Tool 1 ████████░░░░░░░░░░░░░░░░░░░░  2s
                  Tool 2 ████████░░░░░░  2s
                                  Tool 3 ████████  2s
  Total: 6 seconds

Concurrent (Async):
  Tool 1 ████████░░░░░░░░░░░░░░░░░░░░  2s
  Tool 2 ████████░░░░░░░░░░░░░░░░░░░░  2s
  Tool 3 ████████░░░░░░░░░░░░░░░░░░░░  2s
  Total: ~2 seconds
```

### 1.1 The Basics: `async` / `await`

- `async def` declares a **coroutine** (a function that can pause and resume)
- `await` pauses execution until the awaited operation completes
- `asyncio.run()` starts the event loop from synchronous code

In [ ]:
import asyncio
import time

# --- A regular (synchronous) function ---
def sync_fetch_data(source: str) -> str:
    """Simulates a slow API call (blocking)."""
    time.sleep(1)  # Blocks the entire thread
    return f"Data from {source}"

# --- An async coroutine ---
async def async_fetch_data(source: str) -> str:
    """Simulates a slow API call (non-blocking)."""
    await asyncio.sleep(1)  # Yields control while waiting
    return f"Data from {source}"

# --- Running async code ---
async def main():
    result = await async_fetch_data("weather_api")
    print(result)

# In Jupyter, we can use `await` directly (top-level await)
result = await async_fetch_data("weather_api")
print(result)

Data from weather_api
Data from weather_api


### 1.2 `asyncio.gather()` — Running Tasks Concurrently

This is the most important async pattern for agents. When an agent has multiple independent tool calls, it runs them all at once.

In [4]:
import asyncio
import time

async def call_tool(tool_name: str, delay: float = 1.0) -> dict:
    """Simulate calling an agent tool (e.g., web search, database query)."""
    print(f"  ⏳ Starting: {tool_name}")
    await asyncio.sleep(delay)  # Simulate API latency
    print(f"  ✅ Finished: {tool_name}")
    return {"tool": tool_name, "result": f"Result from {tool_name}"}

# --- SEQUENTIAL: One at a time ---
async def run_sequential():
    start = time.perf_counter()
    r1 = await call_tool("web_search", 1.5)
    r2 = await call_tool("database_query", 1.0)
    r3 = await call_tool("calculator", 0.5)
    elapsed = time.perf_counter() - start
    print(f"  ⏱️  Sequential total: {elapsed:.2f}s\n")
    return [r1, r2, r3]

# --- CONCURRENT: All at once ---
async def run_concurrent():
    start = time.perf_counter()
    r1, r2, r3 = await asyncio.gather(
        call_tool("web_search", 1.5),
        call_tool("database_query", 1.0),
        call_tool("calculator", 0.5),
    )
    elapsed = time.perf_counter() - start
    print(f"  ⏱️  Concurrent total: {elapsed:.2f}s\n")
    return [r1, r2, r3]

print("=== Sequential Execution ===")
seq_results = await run_sequential()

print("=== Concurrent Execution ===")
con_results = await run_concurrent()

print("Notice: Concurrent is ~3x faster because all tools run in parallel!")

=== Sequential Execution ===
  ⏳ Starting: web_search
  ✅ Finished: web_search
  ⏳ Starting: database_query
  ✅ Finished: database_query
  ⏳ Starting: calculator
  ✅ Finished: calculator
  ⏱️  Sequential total: 3.01s

=== Concurrent Execution ===
  ⏳ Starting: web_search
  ⏳ Starting: database_query
  ⏳ Starting: calculator
  ✅ Finished: calculator
  ✅ Finished: database_query
  ✅ Finished: web_search
  ⏱️  Concurrent total: 1.50s

Notice: Concurrent is ~3x faster because all tools run in parallel!


### 1.3 Async Generators — Streaming Pattern

When agents stream LLM responses token-by-token, they use async generators. This is how you see text appear word-by-word in ChatGPT.

In [5]:
import asyncio

async def stream_llm_response(prompt: str):
    """
    Simulate streaming tokens from an LLM.
    In real agents, this streams from OpenAI/Anthropic APIs.
    """
    response_tokens = f"The answer to '{prompt}' is that AI agents are autonomous systems.".split()
    
    for token in response_tokens:
        await asyncio.sleep(0.1)  # Simulate network delay per token
        yield token  # Yield one token at a time

# --- Consuming the stream ---
print("Streaming response: ", end="")
async for token in stream_llm_response("What are AI agents?"):
    print(token, end=" ", flush=True)
print("\n\n✅ Stream complete!")

Streaming response: The answer to 'What are AI agents?' is that AI agents are autonomous systems. 

✅ Stream complete!


### 1.4 Practical Pattern: Async Tool Executor

This is a simplified version of what agent frameworks do internally to execute tools.

In [6]:
import asyncio
from typing import Any, Callable
from dataclasses import dataclass

@dataclass
class ToolResult:
    tool_name: str
    success: bool
    result: Any
    duration: float

async def execute_tools_concurrently(
    tool_calls: list[tuple[str, Callable, dict]]
) -> list[ToolResult]:
    """
    Execute multiple tool calls concurrently.
    Each tool_call is (name, async_function, kwargs).
    
    This is what happens inside an agent when the LLM says:
    "I need to call search AND calculator at the same time"
    """
    
    async def _run_one(name: str, func: Callable, kwargs: dict) -> ToolResult:
        start = asyncio.get_event_loop().time()
        try:
            result = await func(**kwargs)
            duration = asyncio.get_event_loop().time() - start
            return ToolResult(name, True, result, duration)
        except Exception as e:
            duration = asyncio.get_event_loop().time() - start
            return ToolResult(name, False, str(e), duration)
    
    # Run all tools concurrently
    results = await asyncio.gather(
        *[_run_one(name, func, kwargs) for name, func, kwargs in tool_calls]
    )
    return results

# --- Define some mock tools ---
async def search_web(query: str) -> str:
    await asyncio.sleep(1.2)
    return f"Found 5 results for '{query}'"

async def query_database(sql: str) -> list:
    await asyncio.sleep(0.8)
    return [{"id": 1, "name": "Widget A"}, {"id": 2, "name": "Widget B"}]

async def get_weather(city: str) -> dict:
    await asyncio.sleep(0.5)
    return {"city": city, "temp": "22°C", "condition": "Sunny"}

# --- Execute all tools concurrently ---
tool_calls = [
    ("web_search", search_web, {"query": "AI agents tutorial"}),
    ("db_query", query_database, {"sql": "SELECT * FROM products LIMIT 5"}),
    ("weather", get_weather, {"city": "San Francisco"}),
]

results = await execute_tools_concurrently(tool_calls)

print("Tool Execution Results:")
print("-" * 50)
for r in results:
    status = "✅" if r.success else "❌"
    print(f"  {status} {r.tool_name}: {r.result} ({r.duration:.2f}s)")

Tool Execution Results:
--------------------------------------------------
  ✅ web_search: Found 5 results for 'AI agents tutorial' (1.20s)
  ✅ db_query: [{'id': 1, 'name': 'Widget A'}, {'id': 2, 'name': 'Widget B'}] (0.80s)
  ✅ weather: {'city': 'San Francisco', 'temp': '22°C', 'condition': 'Sunny'} (0.50s)


### 1.5 Key Async Concepts Summary

| Concept | What It Does | Agent Use Case |
|---------|-------------|----------------|
| `async def` | Declares a coroutine | Define async tools |
| `await` | Pause until result ready | Wait for API response |
| `asyncio.gather()` | Run multiple tasks concurrently | Parallel tool execution |
| `async for` | Iterate over async generator | Stream LLM tokens |
| `asyncio.wait_for()` | Add timeout to async operation | Tool timeout handling |
| `asyncio.Semaphore` | Limit concurrent operations | Rate limiting |

---

# Section 2: Pydantic & Type Hints — Structured Data for Agents

## Why Pydantic Is Critical for Agents

In AI agent systems, **everything** is structured data:
- **Tool schemas** → JSON Schema that tells the LLM what tools are available
- **LLM responses** → Structured JSON that must be parsed reliably
- **Agent state** → Typed objects tracking the agent's progress
- **Memory entries** → Validated records stored in databases

Pydantic gives you:
1. **Validation** — Catch malformed LLM outputs before they crash your agent
2. **Serialization** — Convert Python objects ↔ JSON automatically  
3. **Schema Generation** — Generate JSON schemas that LLMs understand
4. **IDE Support** — Autocomplete and type checking

### 2.1 Pydantic Basics — Models and Validation

In [7]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional
from enum import Enum

# --- Basic Model ---
class ToolCall(BaseModel):
    """Represents a tool call from the LLM."""
    name: str = Field(description="Name of the tool to call")
    arguments: dict = Field(description="Arguments to pass to the tool")
    confidence: float = Field(
        description="LLM's confidence in this tool choice (0-1)", 
        ge=0, le=1  # ge=greater or equal, le=less or equal
    )

# Create from a dict (simulating LLM JSON output)
llm_output = {
    "name": "search_web",
    "arguments": {"query": "latest AI news"},
    "confidence": 0.92
}

tool_call = ToolCall(**llm_output)
print(f"Tool: {tool_call.name}")
print(f"Args: {tool_call.arguments}")
print(f"Confidence: {tool_call.confidence}")
print(f"\nAs JSON: {tool_call.model_dump_json(indent=2)}")

Tool: search_web
Args: {'query': 'latest AI news'}
Confidence: 0.92

As JSON: {
  "name": "search_web",
  "arguments": {
    "query": "latest AI news"
  },
  "confidence": 0.92
}


In [8]:
# --- Validation in Action ---
from pydantic import ValidationError

# This will FAIL — confidence must be 0-1
try:
    bad_call = ToolCall(
        name="search",
        arguments={"query": "test"},
        confidence=1.5  # Invalid!
    )
except ValidationError as e:
    print("❌ Validation Error (caught before agent crashes!):")
    print(e)

print("\n✅ Without Pydantic, this bad data would silently corrupt your agent's state!")

❌ Validation Error (caught before agent crashes!):
1 validation error for ToolCall
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal

✅ Without Pydantic, this bad data would silently corrupt your agent's state!


### 2.2 Complex Models — Agent State and Messages

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
from datetime import datetime

class Message(BaseModel):
    """A single message in the agent's conversation."""
    role: Literal["system", "user", "assistant", "tool"]
    content: str
    timestamp: datetime = Field(default_factory=datetime.now)
    token_count: Optional[int] = None

class AgentState(BaseModel):
    """Complete state of an AI agent at any point in time."""
    agent_id: str
    model: str = "gpt-4o"
    messages: list[Message] = Field(default_factory=list)
    tools_called: list[str] = Field(default_factory=list)
    total_tokens: int = 0
    total_cost: float = 0.0
    iteration: int = 0
    max_iterations: int = 10
    status: Literal["running", "completed", "error", "waiting_for_human"] = "running"
    
    @property
    def is_done(self) -> bool:
        return self.status in ("completed", "error")
    
    @property
    def budget_remaining(self) -> bool:
        return self.iteration < self.max_iterations

# Create an agent state
state = AgentState(
    agent_id="agent-001",
    model="gpt-4o",
    messages=[
        Message(role="system", content="You are a helpful assistant."),
        Message(role="user", content="What's the weather in Tokyo?"),
    ]
)

print(f"Agent: {state.agent_id}")
print(f"Model: {state.model}")
print(f"Messages: {len(state.messages)}")
print(f"Status: {state.status}")
print(f"Can continue: {state.budget_remaining}")
print(f"\nFull state JSON:\n{state.model_dump_json(indent=2)}")

Agent: agent-001
Model: gpt-4o
Messages: 2
Status: running
Can continue: True

Full state JSON:
{
  "agent_id": "agent-001",
  "model": "gpt-4o",
  "messages": [
    {
      "role": "system",
      "content": "You are a helpful assistant.",
      "timestamp": "2026-03-03T21:35:52.601372",
      "token_count": null
    },
    {
      "role": "user",
      "content": "What's the weather in Tokyo?",
      "timestamp": "2026-03-03T21:35:52.601398",
      "token_count": null
    }
  ],
  "tools_called": [],
  "total_tokens": 0,
  "total_cost": 0.0,
  "iteration": 0,
  "max_iterations": 10,
  "status": "running"
}


### 2.3 JSON Schema Generation — How Tool Schemas Work

This is the **most important** Pydantic feature for agents. When you define tools for an LLM, you provide a JSON Schema. Pydantic generates these automatically!

In [10]:
from pydantic import BaseModel, Field
from typing import Optional

# --- Define a tool's parameters as a Pydantic model ---
class SearchParams(BaseModel):
    """Search the web for information."""
    query: str = Field(description="The search query")
    num_results: int = Field(default=5, description="Number of results to return", ge=1, le=20)
    search_type: Optional[str] = Field(
        default="general", 
        description="Type of search: 'general', 'news', 'images'"
    )

# Generate JSON Schema — this is EXACTLY what gets sent to OpenAI/Anthropic!
schema = SearchParams.model_json_schema()

import json
print("Generated JSON Schema (this goes into the LLM's tool definition):")
print(json.dumps(schema, indent=2))

print("\n" + "="*60)
print("This schema tells the LLM:")
print("  - What parameters the tool accepts")
print("  - What types they expect")
print("  - Which are required vs optional")
print("  - Helpful descriptions for each parameter")

Generated JSON Schema (this goes into the LLM's tool definition):
{
  "description": "Search the web for information.",
  "properties": {
    "query": {
      "description": "The search query",
      "title": "Query",
      "type": "string"
    },
    "num_results": {
      "default": 5,
      "description": "Number of results to return",
      "maximum": 20,
      "minimum": 1,
      "title": "Num Results",
      "type": "integer"
    },
    "search_type": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": "general",
      "description": "Type of search: 'general', 'news', 'images'",
      "title": "Search Type"
    }
  },
  "required": [
    "query"
  ],
  "title": "SearchParams",
  "type": "object"
}

This schema tells the LLM:
  - What parameters the tool accepts
  - What types they expect
  - Which are required vs optional
  - Helpful descriptions for each parameter


In [ ]:
# --- Build a complete OpenAI-style tool definition from Pydantic ---

def pydantic_to_openai_tool(model: type[BaseModel]) -> dict:
    """
    Convert a Pydantic model into an OpenAI function calling tool schema.
    This is essentially what LangChain's @tool decorator does internally!
    """
    schema = model.model_json_schema()
    
    # Extract required fields
    required = schema.get("required", [])
    
    # Build the OpenAI tool format
    return {
        "type": "function",
        "function": {
            "name": model.__name__.lower().replace("params", ""),
            "description": model.__doc__ or "No description",
            "parameters": {
                "type": "object",
                "properties": schema.get("properties", {}),
                "required": required,
            }
        }
    }

# Convert our SearchParams to an OpenAI tool
tool_schema = pydantic_to_openai_tool(SearchParams)
print("OpenAI Tool Schema:")
print(json.dumps(tool_schema, indent=2))

print("\n✅ You just built what frameworks do automatically!")
print("   LangChain's @tool, CrewAI's Tool class — all use this pattern.")

name: searchparams
OpenAI Tool Schema:
{
  "type": "function",
  "function": {
    "name": "search",
    "description": "Search the web for information.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "description": "The search query",
          "title": "Query",
          "type": "string"
        },
        "num_results": {
          "default": 5,
          "description": "Number of results to return",
          "maximum": 20,
          "minimum": 1,
          "title": "Num Results",
          "type": "integer"
        },
        "search_type": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": "general",
          "description": "Type of search: 'general', 'news', 'images'",
          "title": "Search Type"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}

✅ You just built what frameworks 

### 2.4 Parsing LLM Outputs with Pydantic

When the LLM returns JSON (via function calling or structured outputs), Pydantic validates it.

In [14]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json

class LLMDecision(BaseModel):
    """The LLM's decision about what to do next."""
    thought: str = Field(description="The LLM's reasoning")
    action: str = Field(description="Tool to call, or 'finish'")
    action_input: dict = Field(description="Arguments for the tool")
    confidence: float = Field(ge=0, le=1)

# Simulate various LLM outputs (some good, some bad)
llm_outputs = [
    # Good output
    '{"thought": "User wants weather, I should use the weather tool", "action": "get_weather", "action_input": {"city": "Paris"}, "confidence": 0.95}',
    # Missing required field
    '{"thought": "I know the answer", "action": "finish"}',
    # Invalid confidence
    '{"thought": "Let me search", "action": "search", "action_input": {"q": "test"}, "confidence": 2.0}',
]

for i, raw_output in enumerate(llm_outputs):
    print(f"\n--- LLM Output {i+1} ---")
    try:
        parsed = json.loads(raw_output)
        decision = LLMDecision(**parsed)
        print(f"  ✅ Valid! Action: {decision.action}, Confidence: {decision.confidence}")
    except json.JSONDecodeError as e:
        print(f"  ❌ Invalid JSON: {e}")
    except ValidationError as e:
        print(f"  ❌ Validation failed: {e.error_count()} error(s)")
        for err in e.errors():
            print(f"     → {err['loc']}: {err['msg']}")

print("\n💡 Pydantic catches bad LLM outputs BEFORE they crash your agent!")


--- LLM Output 1 ---
  ✅ Valid! Action: get_weather, Confidence: 0.95

--- LLM Output 2 ---
  ❌ Validation failed: 2 error(s)
     → ('action_input',): Field required
     → ('confidence',): Field required

--- LLM Output 3 ---
  ❌ Validation failed: 1 error(s)
     → ('confidence',): Input should be less than or equal to 1

💡 Pydantic catches bad LLM outputs BEFORE they crash your agent!


### 2.5 Pydantic Key Concepts Summary

| Feature | What It Does | Agent Use |
|---------|--------------|-----------|
| `BaseModel` | Define structured data | Tool params, agent state, messages |
| `Field()` | Add validation & descriptions | Tool parameter descriptions for LLM |
| `model_dump()` | Convert to dict | Serialize for storage/API |
| `model_dump_json()` | Convert to JSON string | Send to LLM |
| `model_json_schema()` | Generate JSON Schema | Create tool schemas |
| `model_validate()` | Parse from dict with validation | Parse LLM outputs |
| Validators | Custom validation rules | Ensure data integrity |

---

# Section 3: Decorators — Building a Tool Registration System

## Why Decorators Matter for Agents

Every agent framework uses decorators to register tools:

```python
# LangChain
@tool
def search(query: str) -> str:
    """Search the web."""
    ...

# CrewAI  
@tool("Search Tool")
def search(query: str) -> str:
    ...
```

Understanding decorators means you can:
- Build your own tool system
- Debug framework issues
- Create custom middleware (logging, timing, retries)

### 3.1 Decorator Refresher

In [15]:
import functools
import time

# --- Basic decorator: timing ---
def timed(func):
    """Decorator that measures execution time of a function."""
    @functools.wraps(func)  # Preserves original function metadata
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"  ⏱️  {func.__name__} took {elapsed:.3f}s")
        return result
    return wrapper

@timed
def slow_operation(n: int) -> int:
    """Compute sum of range."""
    time.sleep(0.5)  # Simulate slow work
    return sum(range(n))

result = slow_operation(1000)
print(f"  Result: {result}")

# functools.wraps preserves the original function's name and docstring
print(f"  Function name: {slow_operation.__name__}")
print(f"  Docstring: {slow_operation.__doc__}")

  ⏱️  slow_operation took 0.501s
  Result: 499500
  Function name: slow_operation
  Docstring: Compute sum of range.


### 3.2 Building a `@tool` Decorator from Scratch

This is the core pattern used by LangChain, CrewAI, and other frameworks.

In [16]:
import inspect
import functools
import json
from typing import get_type_hints

# Python type → JSON Schema type mapping
TYPE_MAP = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

def tool(func):
    """
    Decorator that converts a Python function into an agent tool.
    
    It automatically extracts:
    - Function name → tool name
    - Docstring → tool description
    - Type hints → JSON schema for parameters
    
    This is a simplified version of what LangChain's @tool does!
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    
    # Extract metadata from the function
    hints = get_type_hints(func)
    sig = inspect.signature(func)
    
    # Build parameter schema
    properties = {}
    required = []
    
    for param_name, param in sig.parameters.items():
        if param_name == "return":
            continue
            
        param_type = hints.get(param_name, str)
        json_type = TYPE_MAP.get(param_type, "string")
        
        properties[param_name] = {
            "type": json_type,
            "description": f"Parameter: {param_name}"
        }
        
        # If no default value, it's required
        if param.default == inspect.Parameter.empty:
            required.append(param_name)
    
    # Attach the schema to the function
    wrapper.tool_schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip(),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            }
        }
    }
    
    wrapper.is_tool = True
    return wrapper

# --- Use the decorator ---

@tool
def search_web(query: str, max_results: int = 5) -> str:
    """Search the web for information on any topic."""
    return f"Found {max_results} results for: {query}"

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_time() -> str:
    """Get the current date and time."""
    from datetime import datetime
    return datetime.now().isoformat()

# Inspect the generated schemas
print("Generated Tool Schemas:")
print("=" * 50)
for t in [search_web, calculate, get_current_time]:
    print(f"\n🔧 {t.__name__}:")
    print(json.dumps(t.tool_schema, indent=2))

Generated Tool Schemas:

🔧 search_web:
{
  "type": "function",
  "function": {
    "name": "search_web",
    "description": "Search the web for information on any topic.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "Parameter: query"
        },
        "max_results": {
          "type": "integer",
          "description": "Parameter: max_results"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}

🔧 calculate:
{
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "Evaluate a mathematical expression and return the result.",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "Parameter: expression"
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}

🔧 get_current_time:
{
  "type": "function",
  "function": {
    "name": "g

### 3.3 Building a Tool Registry

The Tool Registry holds all available tools and provides:
- Registration of new tools
- Schema export (for the LLM)
- Tool execution by name (when the LLM requests it)

In [ ]:
import json
from typing import Callable, Any

class ToolRegistry:
    """
    Registry that manages agent tools.
    
    This is the core of how every agent framework manages tools:
    1. Register tools at startup
    2. Export schemas to the LLM
    3. Execute tools when the LLM requests them
    """
    
    def __init__(self):
        self._tools: dict[str, Callable] = {}
        self._schemas: list[dict] = []
    
    def register(self, func: Callable) -> Callable:
        """Register a tool (can be used as decorator: @registry.register)"""
        if not hasattr(func, 'tool_schema'):
            # If not already decorated with @tool, do it now
            func = tool(func)
        
        name = func.tool_schema["function"]["name"]
        self._tools[name] = func
        self._schemas.append(func.tool_schema)
        print(f"  📝 Registered tool: {name}")
        return func
    
    def get_schemas(self) -> list[dict]:
        """Get all tool schemas (sent to the LLM)."""
        return self._schemas
    
    def execute(self, tool_name: str, arguments: dict) -> str:
        """Execute a tool by name with given arguments."""
        if tool_name not in self._tools:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        
        try:
            result = self._tools[tool_name](**arguments)
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {str(e)}"})
    
    def list_tools(self) -> list[str]:
        """List all registered tool names."""
        return list(self._tools.keys())

# --- Create registry and register tools ---
registry = ToolRegistry()

# Method 1: Register our existing decorated tools
registry.register(search_web)
registry.register(calculate)
registry.register(get_current_time)

# Method 2: Use registry.register as a decorator
@registry.register
def get_weather(city: str) -> str:
    """Get current weather for a specified city."""
    weather = {"London": "15°C, Rainy", "Tokyo": "28°C, Sunny", "Paris": "22°C, Cloudy"}
    return weather.get(city, f"Weather unavailable for {city}")

print(f"\n📋 Registered tools: {registry.list_tools()}")
print(f"\n📄 Tool schemas (sent to LLM):")
print(json.dumps(registry.get_schemas(), indent=2)[:500] + "\n...")

# Execute tools (simulating what happens when LLM returns tool calls)
print("\n=== Executing Tools ===")
print(registry.execute("search_web", {"query": "Python async tutorial"}))
print(registry.execute("calculate", {"expression": "2 ** 10"}))
print(registry.execute("get_weather", {"city": "Tokyo"}))
print(registry.execute("unknown_tool", {}))  # Error handling

---
# Section 4: Error Handling & Retry Logic

## Why This Is Critical for Agents

AI agents interact with **unreliable** external systems:
- LLM APIs → Rate limits, timeouts, server errors
- Web APIs → 5xx errors, network issues
- Databases → Connection drops, query failures
- User tools → Unexpected inputs, runtime errors

An agent that crashes on the first error is **useless**. Production agents need:
1. **Custom exceptions** for different failure types
2. **Retry logic** with exponential backoff
3. **Circuit breakers** to stop hitting broken services
4. **Graceful degradation** when things fail

### 4.1 Custom Exception Hierarchy for Agents

In [17]:
class AgentError(Exception):
    """Base exception for all agent errors."""
    pass

class LLMError(AgentError):
    """Errors from LLM API calls."""
    pass

class RateLimitError(LLMError):
    """Rate limit exceeded on LLM API."""
    def __init__(self, retry_after: float = 60.0):
        self.retry_after = retry_after
        super().__init__(f"Rate limit exceeded. Retry after {retry_after}s")

class ToolExecutionError(AgentError):
    """Error executing a tool."""
    def __init__(self, tool_name: str, original_error: Exception):
        self.tool_name = tool_name
        self.original_error = original_error
        super().__init__(f"Tool '{tool_name}' failed: {original_error}")

class MaxIterationsError(AgentError):
    """Agent exceeded maximum iteration limit."""
    def __init__(self, max_iters: int, current_iter: int):
        self.max_iters = max_iters
        self.current_iter = current_iter
        super().__init__(f"Exceeded max iterations ({current_iter}/{max_iters})")

class GuardrailViolation(AgentError):
    """Agent attempted a blocked action."""
    pass

# --- Using the exception hierarchy ---
def simulate_agent_error(error_type: str):
    if error_type == "rate_limit":
        raise RateLimitError(retry_after=30.0)
    elif error_type == "tool_error":
        raise ToolExecutionError("web_search", ValueError("API key expired"))
    elif error_type == "max_iters":
        raise MaxIterationsError(10, 11)

# Catching errors at different levels
for error_type in ["rate_limit", "tool_error", "max_iters"]:
    try:
        simulate_agent_error(error_type)
    except RateLimitError as e:
        print(f"🔄 Rate limited — waiting {e.retry_after}s before retry")
    except ToolExecutionError as e:
        print(f"🔧 Tool '{e.tool_name}' broke — trying alternative tool")
    except AgentError as e:
        print(f"🤖 Agent error: {e}")

🔄 Rate limited — waiting 30.0s before retry
🔧 Tool 'web_search' broke — trying alternative tool
🤖 Agent error: Exceeded max iterations (11/10)


### 4.2 Retry with Exponential Backoff — Manual Implementation

In [18]:
import asyncio
import random
import time

async def retry_with_backoff(
    func,
    *args,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    jitter: bool = True,
    retryable_exceptions: tuple = (Exception,),
    **kwargs
):
    """
    Retry an async function with exponential backoff.
    
    This is the pattern used in production agents for LLM API calls.
    
    Backoff schedule (with base_delay=1.0):
      Attempt 1: Wait 1s
      Attempt 2: Wait 2s  
      Attempt 3: Wait 4s
      Attempt 4: Wait 8s
      ...with random jitter to prevent thundering herd
    """
    last_exception = None
    
    for attempt in range(max_retries + 1):
        try:
            return await func(*args, **kwargs)
        except retryable_exceptions as e:
            last_exception = e
            
            if attempt == max_retries:
                print(f"  ❌ All {max_retries + 1} attempts failed")
                raise
            
            # Calculate delay with exponential backoff
            delay = min(base_delay * (2 ** attempt), max_delay)
            
            # Add jitter (randomness) to prevent all clients retrying at once
            if jitter:
                delay = delay * (0.5 + random.random())
            
            print(f"  ⚠️  Attempt {attempt + 1} failed: {e}")
            print(f"  ⏳ Retrying in {delay:.1f}s...")
            await asyncio.sleep(delay)
    
    raise last_exception

# --- Test it ---
call_count = 0

async def flaky_llm_call(prompt: str) -> str:
    """Simulates an LLM API that fails intermittently."""
    global call_count
    call_count += 1
    
    if call_count <= 2:  # Fail first 2 times
        raise ConnectionError(f"API timeout (attempt {call_count})")
    
    return f"Response to: {prompt}"

call_count = 0
try:
    result = await retry_with_backoff(
        flaky_llm_call,
        "What is AI?",
        max_retries=3,
        base_delay=0.5,  # Short delays for demo
    )
    print(f"  ✅ Success: {result}")
except ConnectionError:
    print("  ❌ All retries exhausted")

  ⚠️  Attempt 1 failed: API timeout (attempt 1)
  ⏳ Retrying in 0.4s...
  ⚠️  Attempt 2 failed: API timeout (attempt 2)
  ⏳ Retrying in 0.6s...
  ✅ Success: Response to: What is AI?


### 4.3 Using the `tenacity` Library (Production-Ready Retries)

In [23]:
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
    RetryError,
)
import logging

# Simple retry with tenacity
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((ConnectionError, TimeoutError)),
)
def call_llm_api(prompt: str) -> str:
    """
    Call LLM API with automatic retries.
    tenacity handles: backoff, jitter, logging, max attempts
    """
    import random
    if random.random() < 0.6:  # 60% chance of failure
        raise ConnectionError("API temporarily unavailable")
    return f"Response: {prompt}"

# Try it (may succeed or fail depending on random)
try:
    result = call_llm_api("Hello")
    print(f"✅ {result}")
except RetryError as e:
    print(f"❌ All retries failed: {e.last_attempt.exception()}")

print("\n💡 tenacity is the standard library for retries in production Python apps")

❌ All retries failed: API temporarily unavailable

💡 tenacity is the standard library for retries in production Python apps


### 4.4 Circuit Breaker Pattern

When a service is DOWN, stop hammering it. Wait for it to recover.

In [ ]:
import time
from enum import Enum

class CircuitState(Enum):
    CLOSED = "closed"      # Normal operation — requests go through
    OPEN = "open"          # Service is down — reject requests immediately
    HALF_OPEN = "half_open"  # Testing if service recovered

class CircuitBreaker:
    """
    Circuit breaker prevents calling a broken service repeatedly.
    
    States:
      CLOSED → Normal, requests go through
      OPEN → Service broken, reject immediately (fast fail)
      HALF_OPEN → Try one request to see if service recovered
    
    State transitions:
      CLOSED → (N failures) → OPEN → (timeout) → HALF_OPEN → (success) → CLOSED
                                                             → (failure) → OPEN
    """
    
    def __init__(
        self, 
        failure_threshold: int = 3,
        recovery_timeout: float = 30.0,
        name: str = "default"
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.name = name
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0.0
    
    def can_execute(self) -> bool:
        """Check if request should be allowed."""
        if self.state == CircuitState.CLOSED:
            return True
        
        if self.state == CircuitState.OPEN:
            # Check if recovery timeout has passed
            if time.time() - self.last_failure_time >= self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                print(f"  🔄 [{self.name}] Circuit HALF_OPEN — testing recovery")
                return True
            return False
        
        # HALF_OPEN — allow one request
        return True
    
    def record_success(self):
        """Record a successful call."""
        if self.state == CircuitState.HALF_OPEN:
            print(f"  ✅ [{self.name}] Service recovered! Circuit CLOSED")
        self.state = CircuitState.CLOSED
        self.failure_count = 0
    
    def record_failure(self):
        """Record a failed call."""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f"  🔴 [{self.name}] Circuit OPEN — blocking requests for {self.recovery_timeout}s")
    
    def execute(self, func, *args, **kwargs):
        """Execute function with circuit breaker protection."""
        if not self.can_execute():
            raise ConnectionError(f"Circuit breaker [{self.name}] is OPEN — service unavailable")
        
        try:
            result = func(*args, **kwargs)
            self.record_success()
            return result
        except Exception as e:
            self.record_failure()
            raise

# --- Demo ---
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=2, name="OpenAI-API")

call_num = 0
def unstable_api(prompt: str) -> str:
    global call_num
    call_num += 1
    if call_num <= 4:  # First 4 calls fail
        raise ConnectionError(f"API error (call {call_num})")
    return f"Success: {prompt}"

# Make several calls
for i in range(7):
    try:
        result = breaker.execute(unstable_api, "test")
        print(f"  Call {i+1}: ✅ {result}")
    except ConnectionError as e:
        print(f"  Call {i+1}: ❌ {e}")
    time.sleep(0.7)  # Small delay between calls

---
# Section 5: REST APIs — The Language of Agent Tools

## Every Tool Call Is an API Call

When an agent "uses a tool," it's making an HTTP request:
- `search_web("AI agents")` → `GET https://api.search.com/search?q=AI+agents`
- `send_email(to, subject, body)` → `POST https://api.email.com/send`
- `get_weather("Paris")` → `GET https://api.weather.com/v1/current?city=Paris`

Understanding HTTP deeply means understanding how agents interact with the world.

### 5.1 HTTP with `requests` (Synchronous)

In [27]:
import requests
import json

# --- GET Request ---
# Using httpbin.org — a free HTTP testing service
response = requests.get(
    "https://httpbin.org/get",
    params={"query": "AI agents", "limit": 5},
    headers={"User-Agent": "AgentBot/1.0"},
    timeout=10,  # Always set timeouts!
)

print(f"Status: {response.status_code}")
print(f"Headers: {dict(list(response.headers.items())[:3])}")
print(f"Body (JSON): {json.dumps(response.json()['args'], indent=2)}")

Status: 200
Headers: {'Date': 'Wed, 04 Mar 2026 06:05:10 GMT', 'Content-Type': 'application/json', 'Content-Length': '368'}
Body (JSON): {
  "limit": "5",
  "query": "AI agents"
}


In [28]:
# --- POST Request (like sending data to an API) ---
response = requests.post(
    "https://httpbin.org/post",
    json={  # Automatically sets Content-Type: application/json
        "model": "gpt-4o",
        "messages": [
            {"role": "user", "content": "Hello!"}
        ]
    },
    headers={
        "Authorization": "Bearer sk-fake-key-123",
        "Content-Type": "application/json",
    },
    timeout=30,
)

print(f"Status: {response.status_code}")
data = response.json()
# httpbin echoes back what we sent
print(f"Sent JSON: {json.dumps(json.loads(data['data']), indent=2)}")
print(f"Auth header received: {data['headers'].get('Authorization', 'N/A')}")

Status: 200
Sent JSON: {
  "model": "gpt-4o",
  "messages": [
    {
      "role": "user",
      "content": "Hello!"
    }
  ]
}
Auth header received: Bearer sk-fake-key-123


### 5.2 Async HTTP with `httpx` (Preferred for Agents)

`httpx` is the async-native HTTP client — preferred for AI agents because it supports concurrent requests.

In [29]:
import httpx
import asyncio
import time

async def demo_httpx():
    """Demonstrate httpx async client for agent tool calls."""
    
    async with httpx.AsyncClient(timeout=10.0) as client:
        
        # --- Single request ---
        response = await client.get(
            "https://httpbin.org/get",
            params={"tool": "weather", "city": "Paris"}
        )
        print(f"Single request: {response.status_code}")
        
        # --- Concurrent requests (like an agent calling 3 tools at once) ---
        start = time.perf_counter()
        
        # httpbin.org/delay/N waits N seconds before responding
        responses = await asyncio.gather(
            client.get("https://httpbin.org/delay/1"),  # 1s delay
            client.get("https://httpbin.org/delay/1"),  # 1s delay  
            client.get("https://httpbin.org/delay/1"),  # 1s delay
        )
        
        elapsed = time.perf_counter() - start
        print(f"\n3 concurrent requests (each 1s delay): {elapsed:.1f}s total")
        print(f"Status codes: {[r.status_code for r in responses]}")
        print(f"💡 ~1s total instead of ~3s sequential!")

await demo_httpx()

Single request: 200

3 concurrent requests (each 1s delay): 1.7s total
Status codes: [200, 200, 200]
💡 ~1s total instead of ~3s sequential!


### 5.3 Authentication Patterns

Agents need to authenticate with many different services. Here are the common patterns:

In [ ]:
import httpx

# Pattern 1: API Key in Header (OpenAI, Anthropic)
headers_api_key = {
    "Authorization": "Bearer sk-your-api-key",
    "Content-Type": "application/json",
}

# Pattern 2: API Key in Query Parameter (some services)
params_api_key = {
    "api_key": "your-key-here",
    "query": "test",
}

# Pattern 3: Basic Auth (username:password)
# auth = httpx.BasicAuth("username", "password")

# Pattern 4: Custom Header (X-API-Key)
headers_custom = {
    "X-API-Key": "your-key",
}

# --- Best Practice: API Key Management ---
import os

class SecureAPIClient:
    """
    API client that loads keys from environment variables.
    NEVER hardcode API keys in your code!
    """
    
    def __init__(self):
        # Load from environment (set via .env file or export)
        self.openai_key = os.environ.get("OPENAI_API_KEY", "not-set")
        self.anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "not-set")
    
    def get_headers(self, provider: str) -> dict:
        if provider == "openai":
            return {"Authorization": f"Bearer {self.openai_key}"}
        elif provider == "anthropic":
            return {
                "x-api-key": self.anthropic_key,
                "anthropic-version": "2023-06-01",
            }
        raise ValueError(f"Unknown provider: {provider}")

client = SecureAPIClient()
print("OpenAI headers:", client.get_headers("openai"))
print("Anthropic headers:", client.get_headers("anthropic"))
print("\n⚠️  Keys show 'not-set' because env vars aren't configured (that's expected!)")

### 5.4 Rate Limiting — Token Bucket Algorithm

When agents make many API calls, you need rate limiting to avoid getting blocked.

In [30]:
import time
import asyncio

class TokenBucket:
    """
    Token bucket rate limiter.
    
    How it works:
    - Bucket starts with `capacity` tokens
    - Each request consumes 1 token
    - Tokens refill at `rate` per second
    - If no tokens available, wait until one is available
    
    Example: OpenAI allows 500 requests/minute
    → TokenBucket(rate=500/60, capacity=500)
    """
    
    def __init__(self, rate: float, capacity: int):
        self.rate = rate          # Tokens added per second
        self.capacity = capacity  # Maximum tokens
        self.tokens = capacity    # Current tokens
        self.last_refill = time.monotonic()
    
    def _refill(self):
        """Add tokens based on elapsed time."""
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last_refill = now
    
    async def acquire(self):
        """Wait until a token is available, then consume it."""
        while True:
            self._refill()
            if self.tokens >= 1:
                self.tokens -= 1
                return
            # Calculate wait time for next token
            wait_time = (1 - self.tokens) / self.rate
            await asyncio.sleep(wait_time)
    
    @property
    def available(self) -> int:
        self._refill()
        return int(self.tokens)

# --- Demo: 5 requests per second limiter ---
limiter = TokenBucket(rate=5.0, capacity=5)

async def rate_limited_api_call(request_id: int):
    await limiter.acquire()  # Wait for token
    print(f"  Request {request_id} sent at {time.strftime('%H:%M:%S')}.{int(time.time()*100)%100:02d}  (tokens left: {limiter.available})")
    return f"Response {request_id}"

# Fire 10 requests — they'll be rate-limited to 5/second
print("Sending 10 requests with rate limit of 5/sec:")
results = await asyncio.gather(*[rate_limited_api_call(i) for i in range(10)])
print(f"\n✅ All {len(results)} requests completed (rate limited)")

Sending 10 requests with rate limit of 5/sec:
  Request 0 sent at 22:10:13.54  (tokens left: 4)
  Request 1 sent at 22:10:13.55  (tokens left: 3)
  Request 2 sent at 22:10:13.55  (tokens left: 2)
  Request 3 sent at 22:10:13.55  (tokens left: 1)
  Request 4 sent at 22:10:13.55  (tokens left: 0)
  Request 8 sent at 22:10:13.75  (tokens left: 0)
  Request 7 sent at 22:10:13.95  (tokens left: 0)
  Request 9 sent at 22:10:14.15  (tokens left: 0)
  Request 6 sent at 22:10:14.35  (tokens left: 0)
  Request 5 sent at 22:10:14.55  (tokens left: 0)

✅ All 10 requests completed (rate limited)


---
# Section 6: JSON Schemas — The Language of Tool Definitions

## How LLMs Know What Tools Are Available

When you give an LLM tools to use, you provide **JSON Schemas** that describe:
- What the tool does (description)
- What parameters it accepts (properties)
- Which parameters are required
- What types each parameter expects

This is the contract between your agent code and the LLM.

### 6.1 Building OpenAI-Style Tool Schemas by Hand

In [ ]:
import json

# --- OpenAI Function Calling Schema Format ---
# This is the exact format OpenAI/Anthropic/most LLMs expect

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": (
                "Search the web for current information. Use this when the user "
                "asks about recent events, news, or anything that requires "
                "up-to-date information."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query. Be specific and include relevant keywords."
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Number of results to return (1-10)",
                        "default": 5
                    },
                    "search_type": {
                        "type": "string",
                        "description": "Type of search to perform",
                        "enum": ["web", "news", "images", "academic"]
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": (
                "Send an email to a specified recipient. Always confirm with "
                "the user before sending."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {
                        "type": "string",
                        "description": "Recipient email address"
                    },
                    "subject": {
                        "type": "string",
                        "description": "Email subject line"
                    },
                    "body": {
                        "type": "string",
                        "description": "Email body content (supports markdown)"
                    },
                    "urgent": {
                        "type": "boolean",
                        "description": "Whether to mark as urgent",
                        "default": False
                    }
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

print("Complete tool schemas (this is what gets sent to the LLM):")
print(json.dumps(tools, indent=2))

### 6.2 From Pydantic Models to Tool Schemas (The Smart Way)

Instead of writing schemas by hand, let Pydantic generate them:

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
import json

class SearchWebParams(BaseModel):
    """Search the web for current information on any topic."""
    query: str = Field(description="The search query")
    num_results: int = Field(default=5, description="Number of results (1-10)", ge=1, le=10)
    search_type: Literal["web", "news", "images", "academic"] = Field(
        default="web", description="Type of search"
    )

class SendEmailParams(BaseModel):
    """Send an email to a recipient. Always confirm with user first."""
    to: str = Field(description="Recipient email address")
    subject: str = Field(description="Email subject line")
    body: str = Field(description="Email body (supports markdown)")
    urgent: bool = Field(default=False, description="Mark as urgent")

class QueryDatabaseParams(BaseModel):
    """Run a read-only SQL query against the application database."""
    query: str = Field(description="SQL SELECT query to execute")
    database: Literal["users", "products", "orders"] = Field(
        description="Which database to query"
    )
    limit: int = Field(default=100, description="Max rows to return", ge=1, le=1000)

def models_to_tools(*models: type[BaseModel]) -> list[dict]:
    """Convert multiple Pydantic models into OpenAI tool schemas."""
    tools = []
    for model in models:
        schema = model.model_json_schema()
        
        # Remove Pydantic-specific fields that OpenAI doesn't need
        properties = schema.get("properties", {})
        for prop in properties.values():
            prop.pop("title", None)
        
        tool = {
            "type": "function",
            "function": {
                "name": model.__name__.replace("Params", "").lower(),
                "description": (model.__doc__ or "").strip(),
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": schema.get("required", []),
                }
            }
        }
        tools.append(tool)
    
    return tools

# Generate all tool schemas at once
all_tools = models_to_tools(SearchWebParams, SendEmailParams, QueryDatabaseParams)
print(f"Generated {len(all_tools)} tool schemas from Pydantic models:\n")

for t in all_tools:
    print(f"🔧 {t['function']['name']}: {t['function']['description'][:60]}...")
    
print(f"\nFull schema example (search_web):")
print(json.dumps(all_tools[0], indent=2))

### 6.3 JSON Schema Validation

Validate that LLM outputs conform to expected schemas:

In [ ]:
import jsonschema
import json

# Define the schema for an agent's action output
action_schema = {
    "type": "object",
    "properties": {
        "thought": {"type": "string", "minLength": 1},
        "action": {"type": "string", "enum": ["search", "calculate", "finish"]},
        "action_input": {"type": "object"},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
    },
    "required": ["thought", "action", "action_input"],
}

# Test various outputs
test_outputs = [
    {
        "thought": "I need to search for this",
        "action": "search",
        "action_input": {"query": "AI agents"},
        "confidence": 0.9
    },
    {
        "thought": "Done",
        "action": "invalid_action",  # Not in enum!
        "action_input": {}
    },
    {
        "action": "search",  # Missing "thought"!
        "action_input": {}
    },
]

for i, output in enumerate(test_outputs):
    try:
        jsonschema.validate(output, action_schema)
        print(f"Output {i+1}: ✅ Valid — action='{output['action']}'")
    except jsonschema.ValidationError as e:
        print(f"Output {i+1}: ❌ Invalid — {e.message}")

print("\n💡 Schema validation catches malformed LLM outputs before they break your agent")

---
# Section 7: Capstone — Build an AsyncAPIClient

## Putting It All Together

Let's combine everything from this week into a single class that represents
**the tool execution layer of an AI agent**:

- ✅ Async HTTP calls (Section 1 + 5)
- ✅ Pydantic validation (Section 2)
- ✅ Error handling with custom exceptions (Section 4)
- ✅ Retry with exponential backoff (Section 4)
- ✅ Rate limiting (Section 5)
- ✅ Circuit breaker (Section 4)

In [ ]:
import asyncio
import time
import random
import json
from dataclasses import dataclass, field
from typing import Any, Optional
from pydantic import BaseModel, Field

# ============================================================
# Custom Exceptions
# ============================================================

class APIClientError(Exception):
    """Base error for API client."""
    pass

class APIRateLimitError(APIClientError):
    def __init__(self, retry_after: float = 60):
        self.retry_after = retry_after
        super().__init__(f"Rate limited — retry after {retry_after}s")

class APITimeoutError(APIClientError):
    pass

class APIServerError(APIClientError):
    def __init__(self, status_code: int, body: str):
        self.status_code = status_code
        super().__init__(f"Server error {status_code}: {body}")

# ============================================================
# Response Model (Pydantic)
# ============================================================

class APIResponse(BaseModel):
    """Validated API response."""
    status_code: int
    success: bool
    data: Any = None
    error: Optional[str] = None
    latency_ms: float = 0
    retries: int = 0

# ============================================================
# Rate Limiter
# ============================================================

class AsyncRateLimiter:
    def __init__(self, requests_per_second: float = 10):
        self.rate = requests_per_second
        self.tokens = requests_per_second
        self.capacity = requests_per_second
        self.last_refill = time.monotonic()
        self._lock = asyncio.Lock()
    
    async def acquire(self):
        async with self._lock:
            now = time.monotonic()
            elapsed = now - self.last_refill
            self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
            self.last_refill = now
            
            if self.tokens < 1:
                wait = (1 - self.tokens) / self.rate
                await asyncio.sleep(wait)
                self.tokens = 0
            else:
                self.tokens -= 1

# ============================================================
# Circuit Breaker
# ============================================================

class AsyncCircuitBreaker:
    def __init__(self, failure_threshold: int = 5, recovery_time: float = 30):
        self.failure_threshold = failure_threshold
        self.recovery_time = recovery_time
        self.failures = 0
        self.last_failure = 0.0
        self.is_open = False
    
    def check(self):
        if not self.is_open:
            return
        if time.monotonic() - self.last_failure >= self.recovery_time:
            self.is_open = False
            self.failures = 0
        else:
            raise APIClientError("Circuit breaker is OPEN — service unavailable")
    
    def record_failure(self):
        self.failures += 1
        self.last_failure = time.monotonic()
        if self.failures >= self.failure_threshold:
            self.is_open = True
    
    def record_success(self):
        self.failures = 0
        self.is_open = False

# ============================================================
# The Main Client
# ============================================================

class AsyncAPIClient:
    """
    Production-ready async API client for AI agent tool execution.
    
    Features:
    - Async HTTP with concurrent request support
    - Pydantic response validation
    - Retry with exponential backoff
    - Rate limiting (token bucket)
    - Circuit breaker for fault tolerance
    - Request/response logging
    """
    
    def __init__(
        self,
        base_url: str = "",
        api_key: str = "",
        requests_per_second: float = 10,
        max_retries: int = 3,
        timeout: float = 30.0,
    ):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.max_retries = max_retries
        self.timeout = timeout
        self.rate_limiter = AsyncRateLimiter(requests_per_second)
        self.circuit_breaker = AsyncCircuitBreaker()
        self.request_log: list[dict] = []
    
    async def request(
        self,
        method: str,
        path: str,
        json_data: dict = None,
        params: dict = None,
    ) -> APIResponse:
        """
        Make an HTTP request with all protections enabled.
        """
        url = f"{self.base_url}{path}"
        start = time.perf_counter()
        retries = 0
        last_error = None
        
        for attempt in range(self.max_retries + 1):
            try:
                # Check circuit breaker
                self.circuit_breaker.check()
                
                # Wait for rate limit token
                await self.rate_limiter.acquire()
                
                # Simulate HTTP request (in production, use httpx)
                result = await self._simulate_request(method, url, json_data, params)
                
                # Success!
                self.circuit_breaker.record_success()
                latency = (time.perf_counter() - start) * 1000
                
                response = APIResponse(
                    status_code=200,
                    success=True,
                    data=result,
                    latency_ms=latency,
                    retries=retries,
                )
                
                self._log(method, url, response)
                return response
                
            except APIRateLimitError as e:
                retries += 1
                last_error = e
                delay = e.retry_after
                await asyncio.sleep(min(delay, 5))  # Cap for demo
                
            except (APIServerError, APITimeoutError) as e:
                retries += 1
                last_error = e
                self.circuit_breaker.record_failure()
                
                if attempt < self.max_retries:
                    delay = min(1.0 * (2 ** attempt) * (0.5 + random.random()), 30)
                    await asyncio.sleep(delay)
        
        # All retries exhausted
        latency = (time.perf_counter() - start) * 1000
        response = APIResponse(
            status_code=500,
            success=False,
            error=str(last_error),
            latency_ms=latency,
            retries=retries,
        )
        self._log("FAILED " + method, url, response)
        return response
    
    async def _simulate_request(self, method, url, json_data, params):
        """Simulate an HTTP request (replace with httpx in production)."""
        await asyncio.sleep(0.1 + random.random() * 0.2)  # Simulate latency
        
        # Randomly simulate failures (30% chance)
        roll = random.random()
        if roll < 0.1:
            raise APIRateLimitError(retry_after=1.0)
        elif roll < 0.2:
            raise APIServerError(503, "Service temporarily unavailable")
        elif roll < 0.25:
            raise APITimeoutError("Request timed out")
        
        return {"url": url, "method": method, "params": params, "body": json_data}
    
    def _log(self, method: str, url: str, response: APIResponse):
        self.request_log.append({
            "method": method,
            "url": url,
            "status": response.status_code,
            "latency_ms": round(response.latency_ms, 1),
            "retries": response.retries,
        })
    
    def get_stats(self) -> dict:
        """Get client statistics."""
        if not self.request_log:
            return {"total": 0}
        
        total = len(self.request_log)
        successes = sum(1 for r in self.request_log if r["status"] == 200)
        avg_latency = sum(r["latency_ms"] for r in self.request_log) / total
        total_retries = sum(r["retries"] for r in self.request_log)
        
        return {
            "total_requests": total,
            "success_rate": f"{successes/total*100:.1f}%",
            "avg_latency_ms": round(avg_latency, 1),
            "total_retries": total_retries,
        }

# ============================================================
# Demo: Use the client like an agent would
# ============================================================

client = AsyncAPIClient(
    base_url="https://api.example.com",
    api_key="sk-test-key",
    requests_per_second=5,
    max_retries=2,
)

print("🤖 Simulating agent making concurrent tool calls...\n")

# Agent needs to: search + check weather + query database — all at once
results = await asyncio.gather(
    client.request("GET", "/search", params={"q": "AI agents"}),
    client.request("GET", "/weather", params={"city": "Paris"}),
    client.request("POST", "/database/query", json_data={"sql": "SELECT * FROM users"}),
    client.request("GET", "/calendar", params={"date": "today"}),
    client.request("POST", "/email/send", json_data={"to": "user@example.com"}),
)

print("Results:")
for r in results:
    status = "✅" if r.success else "❌"
    retries = f" (retried {r.retries}x)" if r.retries > 0 else ""
    print(f"  {status} {r.status_code} — {r.latency_ms:.0f}ms{retries}")
    if r.data:
        print(f"     Data: {r.data.get('url', 'N/A')}")

print(f"\n📊 Client Stats: {json.dumps(client.get_stats(), indent=2)}")

---
# ✅ Week 1 Complete!

## What You Built & Learned

| Skill | What You Can Do Now |
|-------|---------------------|
| **Async Python** | Run multiple tool calls concurrently, stream LLM responses |
| **Pydantic** | Validate LLM outputs, generate tool schemas, type-safe agent state |
| **Decorators** | Build `@tool` decorators, create tool registries |
| **Error Handling** | Custom exceptions, retry with backoff, circuit breakers |
| **REST APIs** | HTTP clients, authentication, rate limiting |
| **JSON Schemas** | Build OpenAI tool schemas, validate structured outputs |
| **AsyncAPIClient** | Production-grade tool execution layer |

## Next Week Preview: Week 2 — LLM APIs

- OpenAI and Anthropic API deep dive
- Function calling and tool use
- Streaming responses
- Prompt engineering for agents
- Structured outputs (JSON mode)
- Token management and cost tracking

---
*Part of the [AI Agents from Scratch Guide](AI-Agents-From-Scratch-Guide.md)*